# PotholeRisk: Data Preprocessing
This notebook cleans and prepares the Bengaluru accident and traffic datasets for model training.

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder

In [2]:
# Load Bengaluru Accident Dataset

accident_df = pd.read_csv(
    "../dataset/accident/Copy of AccidentReports.csv",
    encoding="latin1"
)

bengaluru_accidents = accident_df[
    accident_df["DISTRICTNAME"].isin(
        ["Bengaluru City", "Bengaluru Dist"]
    )
].copy()

traffic_df = pd.read_csv(
    "../dataset/traffic/Banglore_traffic_Dataset.csv"
)

print("Accident Dataset:", bengaluru_accidents.shape)
print("Traffic Dataset:", traffic_df.shape)

Accident Dataset: (49962, 36)
Traffic Dataset: (8936, 16)


In [3]:
print("Before:", bengaluru_accidents.shape)

bengaluru_accidents.drop_duplicates(inplace=True)

print("After:", bengaluru_accidents.shape)

Before: (49962, 36)
After: (49932, 36)


In [4]:
columns_to_drop = [
    "Lane_Type",
    "Road_Markings",
    "Spot_Conditions",
    "Distance_LandMark_Second",
    "Accident_SpotB"
]

bengaluru_accidents.drop(columns=columns_to_drop, inplace=True)

print(bengaluru_accidents.shape)

(49932, 31)


In [5]:
bengaluru_accidents.isnull().sum().sort_values(ascending=False)

RoadJunction               49932
Side_Walk                  44262
landmark_second            43680
Collision_TypeB            22766
Landmark_first               131
Accident_Description         129
Distance_LandMark_First      121
Accident_Road                 14
Road_Character                 2
Severity                       1
Accident_Classification        1
Main_Cause                     1
Accident_SubLocation           1
Accident_Location              1
Hit_Run                        1
Accident_Spot                  1
Road_Condition                 1
Surface_Type                   1
Surface_Condition              1
Road_Type                      1
Weather                        1
Junction_Control               1
Collision_Type                 1
Crime_No                       0
Year                           0
UNITNAME                       0
DISTRICTNAME                   0
RI                             0
Noofvehicle_involved           0
Latitude                       0
Longitude 

In [6]:
more_columns_to_drop = [
    "RoadJunction",
    "Side_Walk",
    "landmark_second",
    "Collision_TypeB"
]

bengaluru_accidents.drop(columns=more_columns_to_drop, inplace=True)

print("New Shape:", bengaluru_accidents.shape)

New Shape: (49932, 27)


In [12]:
categorical_columns = [
    "Accident_Classification",
    "Accident_Spot",
    "Accident_Location",
    "Accident_SubLocation",
    "Main_Cause",
    "Hit_Run",
    "Severity",
    "Collision_Type",
    "Junction_Control",
    "Road_Character",
    "Road_Type",
    "Surface_Type",
    "Surface_Condition",
    "Road_Condition",
    "Weather",
    "Accident_Road"
]

for col in categorical_columns:
    if col in bengaluru_accidents.columns:
        bengaluru_accidents[col] = bengaluru_accidents[col].fillna(
    bengaluru_accidents[col].mode()[0]
)

In [10]:
landmark_columns = [
    "Landmark_first",
    "Distance_LandMark_First"
]

for col in landmark_columns:
    if col in bengaluru_accidents.columns:
        bengaluru_accidents[col] = bengaluru_accidents[col].fillna(
    bengaluru_accidents[col].mode()[0]
)

In [13]:
bengaluru_accidents.isnull().sum().sort_values(ascending=False)

Accident_Description       129
UNITNAME                     0
DISTRICTNAME                 0
Year                         0
RI                           0
Noofvehicle_involved         0
Accident_Classification      0
Accident_Spot                0
Accident_Location            0
Accident_SubLocation         0
Crime_No                     0
Main_Cause                   0
Hit_Run                      0
Collision_Type               0
Severity                     0
Road_Character               0
Road_Type                    0
Surface_Type                 0
Junction_Control             0
Surface_Condition            0
Road_Condition               0
Accident_Road                0
Weather                      0
Landmark_first               0
Distance_LandMark_First      0
Latitude                     0
Longitude                    0
dtype: int64

In [14]:
bengaluru_accidents.drop(columns=["Accident_Description"], inplace=True)

print(bengaluru_accidents.shape)

(49932, 26)


In [15]:
# Create processed dataset folder if it doesn't exist
import os

os.makedirs("../dataset/processed", exist_ok=True)

# Save cleaned dataset
bengaluru_accidents.to_csv(
    "../dataset/processed/bengaluru_accidents_clean.csv",
    index=False
)

print("Dataset saved successfully!")

Dataset saved successfully!


In [16]:
import pandas as pd

df = pd.read_csv("../dataset/processed/bengaluru_accidents_clean.csv")

print(df.shape)

(49932, 26)


In [17]:
traffic_daily = (
    traffic_df
    .groupby("Date")
    .agg({
        "Traffic Volume": "mean",
        "Average Speed": "mean",
        "Congestion Level": "mean",
        "Travel Time Index": "mean",
        "Road Capacity Utilization": "mean",
        "Traffic_Exposure_Index": "mean"
    })
    .reset_index()
)

traffic_daily.head()

KeyError: "Label(s) ['Traffic_Exposure_Index'] do not exist"

In [18]:
print(traffic_df.columns.tolist())


['Date', 'Area Name', 'Road/Intersection Name', 'Traffic Volume', 'Average Speed', 'Travel Time Index', 'Congestion Level', 'Road Capacity Utilization', 'Incident Reports', 'Environmental Impact', 'Public Transport Usage', 'Traffic Signal Compliance', 'Parking Usage', 'Pedestrian and Cyclist Count', 'Weather Conditions', 'Roadwork and Construction Activity']


In [19]:
from sklearn.preprocessing import MinMaxScaler

traffic_features = [
    "Traffic Volume",
    "Average Speed",
    "Congestion Level",
    "Travel Time Index",
    "Road Capacity Utilization"
]

scaler = MinMaxScaler()

traffic_df[traffic_features] = scaler.fit_transform(
    traffic_df[traffic_features]
)

traffic_df["Traffic_Exposure_Index"] = (
      0.30 * traffic_df["Traffic Volume"]
    + 0.25 * traffic_df["Congestion Level"]
    + 0.20 * traffic_df["Travel Time Index"]
    + 0.15 * traffic_df["Road Capacity Utilization"]
    + 0.10 * (1 - traffic_df["Average Speed"])
)

In [20]:
for col in [
    "Date",
    "Traffic Volume",
    "Average Speed",
    "Congestion Level",
    "Travel Time Index",
    "Road Capacity Utilization",
    "Traffic_Exposure_Index"
]:
    print(col, "->", col in traffic_df.columns)

Date -> True
Traffic Volume -> True
Average Speed -> True
Congestion Level -> True
Travel Time Index -> True
Road Capacity Utilization -> True
Traffic_Exposure_Index -> True


In [21]:
traffic_df[
    [
        "Traffic Volume",
        "Average Speed",
        "Congestion Level",
        "Travel Time Index",
        "Road Capacity Utilization",
        "Traffic_Exposure_Index"
    ]
].dtypes

Traffic Volume               float64
Average Speed                float64
Congestion Level             float64
Travel Time Index            float64
Road Capacity Utilization    float64
Traffic_Exposure_Index       float64
dtype: object

In [22]:
traffic_df.groupby("Date").size().head()

Date
2022-01-01    12
2022-01-02    11
2022-01-03     7
2022-01-04    11
2022-01-05    10
dtype: int64

In [23]:
traffic_df.groupby("Date")["Traffic Volume"].mean().head()

Date
2022-01-01    0.462417
2022-01-02    0.349170
2022-01-03    0.330759
2022-01-04    0.373693
2022-01-05    0.420398
Name: Traffic Volume, dtype: float64

In [24]:
traffic_df.groupby("Date")["Traffic_Exposure_Index"].mean().head()


Date
2022-01-01    0.740364
2022-01-02    0.642052
2022-01-03    0.657833
2022-01-04    0.658547
2022-01-05    0.688524
Name: Traffic_Exposure_Index, dtype: float64

In [25]:
traffic_daily = pd.DataFrame()

traffic_daily["Traffic Volume"] = (
    traffic_df.groupby("Date")["Traffic Volume"].mean()
)

traffic_daily["Average Speed"] = (
    traffic_df.groupby("Date")["Average Speed"].mean()
)

traffic_daily["Congestion Level"] = (
    traffic_df.groupby("Date")["Congestion Level"].mean()
)

traffic_daily["Travel Time Index"] = (
    traffic_df.groupby("Date")["Travel Time Index"].mean()
)

traffic_daily["Road Capacity Utilization"] = (
    traffic_df.groupby("Date")["Road Capacity Utilization"].mean()
)

traffic_daily["Traffic_Exposure_Index"] = (
    traffic_df.groupby("Date")["Traffic_Exposure_Index"].mean()
)

traffic_daily = traffic_daily.reset_index()

traffic_daily.head()

,Date,Traffic Volume,Average Speed,Congestion Level,Travel Time Index,Road Capacity Utilization,Traffic_Exposure_Index
0,2022-01-01,0.462417,0.295830,0.859218,0.898976,0.910813,0.740364
1,2022-01-02,0.349170,0.299766,0.755106,0.731533,0.881297,0.642052
2,2022-01-03,0.330759,0.232309,0.757889,0.843489,0.824444,0.657833
3,2022-01-04,0.373693,0.277783,0.764247,0.731388,0.912522,0.658547
4,2022-01-05,0.420398,0.269256,0.811032,0.728321,0.939385,0.688524


In [26]:
print(traffic_daily.shape)

(952, 7)


In [27]:
print("Unique dates:", traffic_df["Date"].nunique())

Unique dates: 952


In [28]:
print(traffic_df["Date"].head(20))

0     2022-01-01
1     2022-01-01
2     2022-01-01
3     2022-01-01
4     2022-01-01
5     2022-01-01
6     2022-01-01
7     2022-01-01
8     2022-01-01
9     2022-01-01
10    2022-01-01
11    2022-01-01
12    2022-01-02
13    2022-01-02
14    2022-01-02
15    2022-01-02
16    2022-01-02
17    2022-01-02
18    2022-01-02
19    2022-01-02
Name: Date, dtype: str


In [29]:
print(traffic_df["Date"].tail(20))


8916    2024-08-07
8917    2024-08-08
8918    2024-08-08
8919    2024-08-08
8920    2024-08-08
8921    2024-08-08
8922    2024-08-08
8923    2024-08-08
8924    2024-08-08
8925    2024-08-08
8926    2024-08-08
8927    2024-08-09
8928    2024-08-09
8929    2024-08-09
8930    2024-08-09
8931    2024-08-09
8932    2024-08-09
8933    2024-08-09
8934    2024-08-09
8935    2024-08-09
Name: Date, dtype: str


In [30]:
traffic_df["Date"].sample(20)

3767    2023-02-07
4713    2023-05-21
319     2022-02-04
6795    2023-12-27
7700    2024-03-30
2611    2022-10-05
1010    2022-04-18
1874    2022-07-19
5801    2023-09-13
596     2022-03-05
7446    2024-03-03
1861    2022-07-17
5549    2023-08-18
4270    2023-04-02
7180    2024-02-04
6980    2024-01-15
8905    2024-08-06
4299    2023-04-05
1581    2022-06-18
5343    2023-07-28
Name: Date, dtype: str

In [31]:
traffic_2022 = traffic_df[
    traffic_df["Date"].str.startswith("2022")
].copy()

print(traffic_2022.shape)

(3424, 17)


In [32]:
traffic_daily = (
    traffic_2022
    .groupby("Date")
    .agg({
        "Traffic Volume": "mean",
        "Average Speed": "mean",
        "Congestion Level": "mean",
        "Travel Time Index": "mean",
        "Road Capacity Utilization": "mean",
        "Traffic_Exposure_Index": "mean"
    })
    .reset_index()
)

traffic_daily.head()

,Date,Traffic Volume,Average Speed,Congestion Level,Travel Time Index,Road Capacity Utilization,Traffic_Exposure_Index
0,2022-01-01,0.462417,0.295830,0.859218,0.898976,0.910813,0.740364
1,2022-01-02,0.349170,0.299766,0.755106,0.731533,0.881297,0.642052
2,2022-01-03,0.330759,0.232309,0.757889,0.843489,0.824444,0.657833
3,2022-01-04,0.373693,0.277783,0.764247,0.731388,0.912522,0.658547
4,2022-01-05,0.420398,0.269256,0.811032,0.728321,0.939385,0.688524


In [33]:
print(traffic_daily.shape)

(365, 7)


In [35]:
print(accident_df.columns.tolist())

['DISTRICTNAME', 'UNITNAME', 'Crime_No', 'Year', 'RI', 'Noofvehicle_involved', 'Accident_Classification', 'Accident_Spot', 'Accident_Location', 'Accident_SubLocation', 'Accident_SpotB', 'Main_Cause', 'Hit_Run', 'Severity', 'Collision_Type', 'Junction_Control', 'Road_Character', 'Road_Type', 'Surface_Type', 'Surface_Condition', 'Road_Condition', 'Weather', 'Lane_Type', 'Road_Markings', 'Spot_Conditions', 'Side_Walk', 'RoadJunction', 'Collision_TypeB', 'Accident_Road', 'Landmark_first', 'landmark_second', 'Distance_LandMark_First', 'Distance_LandMark_Second', 'Accident_Description', 'Latitude', 'Longitude']
